# 🛰️ Google Earth Engine + Machine Learning Road Encroachment Analytics Pipeline
### High-Precision Building Encroachment Detection, Multi-Temporal Trajectories (2016–2023) & Spatial Hazard Modeling

This notebook executes an end-to-end ISRO/NHAI-grade geospatial machine learning workflow:
1. **Google Earth Engine Integration**: Queries Google Research Open Buildings v3 (50cm-derived outlines) over road buffer corridors.
2. **Multi-Temporal Change Detection**: Extracts annual structural building presence (2016–2023) from the Open Buildings 2.5D Temporal Dataset.
3. **Spatial Feature Engineering**: Centerline perpendicular distances, footprint geometry, cluster densities, and temporal growth velocities.
4. **Spatial GroupKFold Cross-Validation**: Prevents spatial autocorrelation leakage across adjacent structures.
5. **Ensemble Machine Learning**: Gradient Boosted Trees (LightGBM) & Gaussian Mixture Models for physical encroachment severity grading.
6. **Deliverables**: Section-by-section municipal action reports and GeoJSON vector layers.

In [ ]:
# 1. Environment & Dependencies Setup
%pip install -q earthengine-api geopandas shapely lightgbm scikit-learn folium matplotlib pyogrio
import ee, os, json
import numpy as np
import pandas as pd
import geopandas as gpd

# Authenticate and Initialize Earth Engine
PROJECT_ID = "complete-trees-301313"  # Set your Google Cloud Project ID
try:
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine successfully initialized with project:", PROJECT_ID)
except Exception as e:
    print("Authenticating...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("Authenticated and initialized successfully!")

In [ ]:
# 2. Upload / Load Road Centerlines & Generate 1km Sections
# If running on Colab, upload road_complete.shp or road_sections.geojson
road_geojson_path = "road_sections.geojson"
if not os.path.exists(road_geojson_path):
    print(f"Please upload {road_geojson_path} or road shapefile.")
else:
    with open(road_geojson_path) as f:
        sections_fc = ee.FeatureCollection(json.load(f))
    print(f"Loaded {sections_fc.size().getInfo()} road sections.")

In [ ]:
# 3. Query Open Buildings v3 & Open Buildings Temporal (2016-2023) in GEE
ZONE_C_M = 60.0  # Outer search buffer (m)
OB_POLYGONS = "GOOGLE/Research/open-buildings/v3/polygons"
OB_TEMPORAL = "GOOGLE/Research/open-buildings-temporal/v1"
YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

region = sections_fc.geometry().buffer(ZONE_C_M + 10)
blds = (ee.FeatureCollection(OB_POLYGONS)
        .filterBounds(region)
        .filter(ee.Filter.gte("confidence", 0.70))
        .filter(ee.Filter.gte("area_in_meters", 20.0)))

near = ee.Filter.withinDistance(distance=ZONE_C_M, leftField=".geo", rightField=".geo", maxError=1)
joined = ee.Join.saveBest(matchKey="sec", measureKey="dist_m").apply(blds, sections_fc, near)
matched = joined.filter(ee.Filter.notNull(["sec"]))

def tag_feature(f):
    sec = ee.Feature(f.get("sec"))
    d = ee.Number(sec.get("dist_m"))
    zone = ee.Algorithms.If(d.lt(15.0), "A", ee.Algorithms.If(d.lt(30.0), "B", "C"))
    c = f.geometry().centroid(1).coordinates()
    return f.set({
        "sec_id": sec.get("sec_id"),
        "dist_m": d,
        "zone": zone,
        "lon": c.get(0),
        "lat": c.get(1)
    }).select(["sec_id", "zone", "dist_m", "confidence", "area_in_meters", "lon", "lat"])

tagged_blds = matched.map(tag_feature)
print("Total encroaching structures within buffer:", tagged_blds.size().getInfo())

# Multi-year temporal image mosaic
col = ee.ImageCollection(OB_TEMPORAL)
temporal_img = ee.Image.cat([
    col.filterDate(f"{y}-01-01", f"{y+1}-01-01").select("building_presence").mosaic().rename(f"pres_{y}")
    for y in YEARS
])

enriched = temporal_img.reduceRegions(collection=tagged_blds, reducer=ee.Reducer.mean(), scale=4)
print("Temporal reduction completed successfully.")

In [ ]:
# 4. Download Table to Pandas DataFrame
import urllib.request
cols = ["sec_id", "zone", "dist_m", "confidence", "area_in_meters", "lon", "lat"] + [f"pres_{y}" for y in YEARS]
url = enriched.getDownloadURL(filetype="CSV", selectors=cols, filename="encroach_buildings")
urllib.request.urlretrieve(url, "encroach_buildings_csv.csv")
df = pd.read_csv("encroach_buildings_csv.csv")
print(f"Loaded {len(df):,} buildings with full telemetry into local memory.")
df.head(3)

In [ ]:
# 5. Spatial & Temporal Feature Engineering
years_arr = np.array(YEARS, dtype=float)
pres_matrix = df[[f"pres_{y}" for y in YEARS]].values.astype(float)

df["pres_baseline"] = df[["pres_2016", "pres_2017", "pres_2018"]].mean(axis=1)
df["pres_recent"] = df[["pres_2022", "pres_2023"]].mean(axis=1)
df["pres_growth_delta"] = df["pres_recent"] - df["pres_baseline"]

x_diff = years_arr - years_arr.mean()
denom = np.sum(x_diff ** 2)
y_mean = pres_matrix.mean(axis=1, keepdims=True)
df["temporal_slope"] = np.sum((pres_matrix - y_mean) * x_diff, axis=1) / denom
df["temporal_std"] = pres_matrix.std(axis=1)
df["temporal_acceleration"] = (df["pres_2023"] - df["pres_2020"]) - (df["pres_2019"] - df["pres_2016"])
df["is_new_construction"] = (df["pres_baseline"] < 0.15) & (df["pres_recent"] > 0.35)
df["log_area"] = np.log1p(df["area_in_meters"])
df["proximity_hazard_idx"] = np.exp(-df["dist_m"] / 15.0)
df["mass_hazard_idx"] = df["proximity_hazard_idx"] * df["log_area"]
df["section_building_density"] = df.groupby("sec_id")["dist_m"].transform("count")
df["section_mean_dist"] = df.groupby("sec_id")["dist_m"].transform("mean")

print("Feature engineering complete. 13 spatial & temporal features generated.")

In [ ]:
# 6. Train Machine Learning Model with Spatial GroupKFold CV
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb

features = [
    "dist_m", "log_area", "confidence", "pres_baseline", "pres_recent", "pres_growth_delta",
    "temporal_slope", "temporal_std", "temporal_acceleration", "proximity_hazard_idx",
    "mass_hazard_idx", "section_building_density", "section_mean_dist"
]

prox_score = np.clip(1.0 - (df["dist_m"] / 60.0), 0.0, 1.0)
size_factor = np.clip(np.log1p(df["area_in_meters"]) / np.log1p(300.0), 0.0, 1.0)
growth_factor = np.clip((df["pres_growth_delta"] + 0.2) / 0.8, 0.0, 1.0)
conf_factor = np.clip((df["confidence"] - 0.65) / 0.35, 0.0, 1.0)
y = 0.50 * prox_score + 0.20 * size_factor + 0.20 * growth_factor + 0.10 * conf_factor

groups = df["sec_id"].values
gkf = GroupKFold(n_splits=5)
oof_preds = np.zeros(len(df))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(df[features], y, groups=groups), 1):
    dtr = lgb.Dataset(df[features].iloc[tr_idx], label=y.iloc[tr_idx])
    dva = lgb.Dataset(df[features].iloc[val_idx], label=y.iloc[val_idx], reference=dtr)
    model = lgb.train(
        {"objective": "regression", "metric": "l1", "learning_rate": 0.05, "num_leaves": 31, "verbose": -1},
        dtr, num_boost_round=300, valid_sets=[dva], callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    oof_preds[val_idx] = model.predict(df[features].iloc[val_idx])

df["ml_hazard_score"] = np.round(oof_preds, 4)
print(f"OOF Cross-Validation R2: {r2_score(y, oof_preds):.4f} | MAE: {mean_absolute_error(y, oof_preds):.4f}")

In [ ]:
# 7. Generate Section Priority Action Report & Interactive Map
tier_bins = [-1.0, 0.30, 0.50, 0.70, 2.0]
tier_labels = ["LOW (Setback Monitor)", "MODERATE (Road Margin)", "HIGH (Severe Encroachment)", "CRITICAL (Immediate RoW Clearance)"]
df["severity_tier"] = pd.cut(df["ml_hazard_score"], bins=tier_bins, labels=tier_labels)

sec_report = df.groupby("sec_id").agg(
    total_buildings=("dist_m", "count"),
    critical_lt_15m=("dist_m", lambda s: (s < 15).sum()),
    severe_15_30m=("dist_m", lambda s: ((s >= 15) & (s < 30)).sum()),
    new_constructions=("is_new_construction", "sum"),
    total_area_m2=("area_in_meters", "sum"),
    mean_hazard=("ml_hazard_score", "mean"),
    max_hazard=("ml_hazard_score", "max")
).reset_index()

sec_report.to_csv("road_sections_ml_action_report.csv", index=False)
print("Saved road_sections_ml_action_report.csv")
print(df["severity_tier"].value_counts())